# 🌞 Solar Panel Angle Optimization for Goa

This notebook calculates optimal solar panel angles based on:
- **Location**: Goa, India (Latitude: 15.4909°N, Longitude: 73.8278°E)
- **Time of Day**: Hourly calculations
- **Month/Season**: Seasonal adjustments

**Output**: CSV file with recommended tilt angles for implementation team

## 1. Install & Import Libraries

In [ ]:
# Install pvlib - industry standard for solar calculations
!pip install pvlib

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pvlib
from pvlib import solarposition, irradiance
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots

## 2. Define Location Parameters

**Goa University Campus Location**

In [ ]:
# Goa Location Coordinates
LATITUDE = 15.4909    # degrees North
LONGITUDE = 73.8278   # degrees East
ALTITUDE = 50         # meters above sea level (approximate)
TIMEZONE = 'Asia/Kolkata'

print(f"📍 Location: Goa, India")
print(f"   Latitude:  {LATITUDE}°N")
print(f"   Longitude: {LONGITUDE}°E")
print(f"   Timezone:  {TIMEZONE}")

## 3. Generate Time Range for Calculations

We'll calculate solar positions for every hour of a full year to capture seasonal variations.

In [ ]:
# Generate hourly timestamps for full year 2024
times = pd.date_range(
    start='2024-01-01 00:00:00',
    end='2024-12-31 23:00:00',
    freq='1H',
    tz=TIMEZONE
)

print(f"Generated {len(times)} hourly timestamps")
print(f"From: {times[0]}")
print(f"To:   {times[-1]}")

## 4. Calculate Solar Position (Altitude & Azimuth)

Using **pvlib** to compute:
- **Solar Altitude (Elevation)**: Angle of sun above horizon (0° = horizon, 90° = directly overhead)
- **Solar Azimuth**: Compass direction of sun (0°/360° = North, 90° = East, 180° = South, 270° = West)

In [ ]:
# Calculate solar position for all timestamps
solar_position = solarposition.get_solarposition(
    time=times,
    latitude=LATITUDE,
    longitude=LONGITUDE,
    altitude=ALTITUDE,
    method='nrel_numpy'  # High accuracy method
)

# Create main dataframe
df_solar = pd.DataFrame({
    'Datetime': times,
    'Solar_Elevation': solar_position['elevation'].values,  # Altitude angle
    'Solar_Azimuth': solar_position['azimuth'].values,
    'Solar_Zenith': solar_position['zenith'].values  # 90 - elevation
})

# Add time components
df_solar['Hour'] = df_solar['Datetime'].dt.hour
df_solar['Month'] = df_solar['Datetime'].dt.month
df_solar['Month_Name'] = df_solar['Datetime'].dt.strftime('%B')
df_solar['Day_of_Year'] = df_solar['Datetime'].dt.dayofyear

df_solar.head(10)

## 5. Calculate Optimal Tilt Angle

### Formulas Used:

**General Rule for Fixed Tilt (Annual):**
- Optimal Tilt ≈ Latitude (for Goa: ~15°)

**Seasonal Adjustments:**
- **Winter (Oct-Feb)**: Tilt = Latitude + 15° (sun is lower)
- **Summer (Mar-Sep)**: Tilt = Latitude - 15° (sun is higher)
- **Equinox**: Tilt = Latitude

**For Maximum Energy at Any Moment:**
- Optimal Tilt = 90° - Solar Elevation (to face sun directly)

In [ ]:
def calculate_optimal_tilt(row, latitude=LATITUDE):
    """
    Calculate optimal tilt angle based on month and solar position.
    
    Returns:
    - optimal_tilt_fixed: Best fixed angle for the month
    - optimal_tilt_tracking: Ideal angle to directly face sun (for tracking systems)
    """
    month = row['Month']
    elevation = row['Solar_Elevation']
    
    # Fixed tilt recommendations by season
    if month in [11, 12, 1, 2]:  # Winter months
        fixed_tilt = latitude + 15
        season = 'Winter'
    elif month in [5, 6, 7, 8]:  # Summer/Monsoon months
        fixed_tilt = max(latitude - 15, 0)  # Don't go negative
        season = 'Summer'
    else:  # Spring/Autumn (Mar, Apr, Sep, Oct)
        fixed_tilt = latitude
        season = 'Equinox'
    
    # Tracking tilt (to directly face sun) - only meaningful when sun is up
    if elevation > 0:
        tracking_tilt = 90 - elevation
    else:
        tracking_tilt = np.nan  # Sun below horizon
    
    return pd.Series({
        'Optimal_Tilt_Fixed': fixed_tilt,
        'Optimal_Tilt_Tracking': tracking_tilt,
        'Season': season
    })

# Apply calculation
tilt_results = df_solar.apply(calculate_optimal_tilt, axis=1)
df_solar = pd.concat([df_solar, tilt_results], axis=1)

# Optimal azimuth for Northern Hemisphere is always ~180° (facing South)
df_solar['Optimal_Azimuth'] = 180  # Panels should face South in Northern Hemisphere

df_solar.head(20)

## 6. Filter Daylight Hours Only

Remove nighttime data (when sun is below horizon)

In [ ]:
# Filter for daylight hours (sun above horizon)
df_daylight = df_solar[df_solar['Solar_Elevation'] > 0].copy()

print(f"Total records: {len(df_solar)}")
print(f"Daylight records: {len(df_daylight)}")
print(f"Nighttime records removed: {len(df_solar) - len(df_daylight)}")

## 7. Create Summary by Month (For CSV Export)

This creates a practical monthly recommendation table.

In [ ]:
# Monthly summary statistics
monthly_summary = df_daylight.groupby(['Month', 'Month_Name', 'Season']).agg({
    'Solar_Elevation': ['mean', 'max'],
    'Optimal_Tilt_Fixed': 'first',  # Same for whole month
    'Optimal_Tilt_Tracking': 'mean',
}).round(2)

monthly_summary.columns = ['Avg_Solar_Elevation', 'Max_Solar_Elevation', 
                           'Recommended_Fixed_Tilt', 'Avg_Tracking_Tilt']
monthly_summary = monthly_summary.reset_index()

# Add optimal azimuth (always South = 180° for Northern Hemisphere)
monthly_summary['Recommended_Azimuth'] = 180

# Add notes
def add_notes(row):
    if row['Season'] == 'Winter':
        return 'Sun lower in sky - steeper tilt captures more energy'
    elif row['Season'] == 'Summer':
        return 'Sun higher in sky - flatter tilt optimal; monsoon may reduce output'
    else:
        return 'Transitional period - moderate tilt angle'

monthly_summary['Notes'] = monthly_summary.apply(add_notes, axis=1)

monthly_summary

## 8. Create Hourly Recommendations (Detailed CSV)

In [ ]:
# Hourly averages by month (for more granular control)
hourly_monthly = df_daylight.groupby(['Month', 'Month_Name', 'Hour']).agg({
    'Solar_Elevation': 'mean',
    'Solar_Azimuth': 'mean',
    'Optimal_Tilt_Fixed': 'first',
    'Optimal_Tilt_Tracking': 'mean',
    'Season': 'first'
}).round(2).reset_index()

hourly_monthly['Recommended_Azimuth'] = 180

print(f"Hourly-Monthly recommendations: {len(hourly_monthly)} records")
hourly_monthly.head(20)

## 9. 📤 Export CSV Files for Implementation Team

In [ ]:
# === CSV 1: Monthly Summary (Simple) ===
monthly_export = monthly_summary[['Month', 'Month_Name', 'Season', 
                                   'Recommended_Fixed_Tilt', 'Recommended_Azimuth',
                                   'Avg_Solar_Elevation', 'Max_Solar_Elevation', 'Notes']]

monthly_export.to_csv('optimal_angles_monthly.csv', index=False)
print("✅ Saved: optimal_angles_monthly.csv")

# === CSV 2: Hourly by Month (Detailed) ===
hourly_export = hourly_monthly[['Month', 'Month_Name', 'Hour', 'Season',
                                 'Solar_Elevation', 'Solar_Azimuth',
                                 'Optimal_Tilt_Fixed', 'Optimal_Tilt_Tracking',
                                 'Recommended_Azimuth']]
hourly_export.columns = ['Month', 'Month_Name', 'Hour', 'Season',
                         'Avg_Solar_Elevation', 'Avg_Solar_Azimuth',
                         'Recommended_Fixed_Tilt', 'Optimal_Tracking_Tilt',
                         'Recommended_Panel_Azimuth']

hourly_export.to_csv('optimal_angles_hourly.csv', index=False)
print("✅ Saved: optimal_angles_hourly.csv")

# === CSV 3: Full Year Daily Noon Values ===
noon_data = df_daylight[df_daylight['Hour'] == 12][['Datetime', 'Month_Name', 'Season',
                                                     'Solar_Elevation', 'Solar_Azimuth',
                                                     'Optimal_Tilt_Fixed', 'Optimal_Tilt_Tracking',
                                                     'Optimal_Azimuth']].copy()
noon_data['Date'] = noon_data['Datetime'].dt.date
noon_data.to_csv('optimal_angles_daily_noon.csv', index=False)
print("✅ Saved: optimal_angles_daily_noon.csv")

print("\n📁 Files ready to send to implementation team!")

## 10. 📊 Visualizations

In [ ]:
# Solar Elevation Throughout the Year (at noon)
fig = px.line(
    noon_data, 
    x='Date', 
    y='Solar_Elevation',
    color='Season',
    title='Solar Elevation at Noon Throughout 2024 (Goa)',
    labels={'Solar_Elevation': 'Solar Elevation (degrees)', 'Date': 'Date'}
)
fig.update_layout(width=900, height=500)
fig.show()

In [ ]:
# Recommended Tilt Angle by Month
fig = px.bar(
    monthly_summary,
    x='Month_Name',
    y='Recommended_Fixed_Tilt',
    color='Season',
    title='Recommended Solar Panel Tilt Angle by Month (Goa)',
    labels={'Recommended_Fixed_Tilt': 'Tilt Angle (degrees)', 'Month_Name': 'Month'}
)
fig.add_hline(y=LATITUDE, line_dash="dash", line_color="red", 
              annotation_text=f"Latitude ({LATITUDE}°)")
fig.update_layout(width=900, height=500)
fig.show()

In [ ]:
# Sun Path Diagram (Polar Plot)
# Sample one day per month at noon
sample_days = df_daylight[
    (df_daylight['Datetime'].dt.day == 15) & 
    (df_daylight['Hour'] >= 6) & 
    (df_daylight['Hour'] <= 18)
].copy()

fig = px.scatter_polar(
    sample_days,
    r=90 - sample_days['Solar_Elevation'],  # Convert to zenith for better visualization
    theta=sample_days['Solar_Azimuth'],
    color='Month_Name',
    title='Sun Path Diagram - Goa (15th of each month)',
    range_r=[0, 90]
)
fig.update_layout(width=700, height=700)
fig.show()

---

# 📥 PHASE 2: After Implementation - Before/After Comparison

**Instructions:** 
1. Send the CSV files above to your implementation team
2. Once they implement the new angles, they will send you:
   - `initial_readings.csv` - Power output BEFORE angle optimization
   - `final_readings.csv` - Power output AFTER angle optimization
3. Load those files below and run the comparison visualizations

In [ ]:
# === PLACEHOLDER: Load Before/After Data ===
# Uncomment and modify paths when you receive the data

# df_initial = pd.read_csv('initial_readings.csv')
# df_final = pd.read_csv('final_readings.csv')

# For now, we can use existing aMESS data as "initial" baseline
print("⏳ Waiting for implementation team data...")
print("   Expected files:")
print("   - initial_readings.csv (before optimization)")
print("   - final_readings.csv (after optimization)")

In [ ]:
# === TEMPLATE: Before/After Comparison Visualization ===

def compare_before_after(df_before, df_after, power_column='average_dc_power'):
    """
    Create comparison visualizations for before/after optimization.
    
    Parameters:
    - df_before: DataFrame with initial readings
    - df_after: DataFrame with optimized readings  
    - power_column: Name of the power output column
    """
    
    # Calculate improvement metrics
    total_before = df_before[power_column].sum()
    total_after = df_after[power_column].sum()
    improvement = ((total_after - total_before) / total_before) * 100
    
    print("=" * 50)
    print("📊 OPTIMIZATION RESULTS")
    print("=" * 50)
    print(f"Total Energy BEFORE: {total_before:,.2f} Wh")
    print(f"Total Energy AFTER:  {total_after:,.2f} Wh")
    print(f"Improvement: {improvement:+.2f}%")
    print("=" * 50)
    
    return improvement

# Example usage (uncomment when data is available):
# improvement = compare_before_after(df_initial, df_final)

In [ ]:
# === TEMPLATE: Time Series Comparison Plot ===

def plot_comparison_timeseries(df_before, df_after, datetime_col='Datetime', power_col='average_dc_power'):
    """
    Plot before/after power output over time.
    """
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_before[datetime_col],
        y=df_before[power_col],
        name='Before Optimization',
        line=dict(color='red', dash='dot'),
        opacity=0.7
    ))
    
    fig.add_trace(go.Scatter(
        x=df_after[datetime_col],
        y=df_after[power_col],
        name='After Optimization',
        line=dict(color='green'),
        opacity=0.9
    ))
    
    fig.update_layout(
        title='Solar Power Output: Before vs After Angle Optimization',
        xaxis_title='Time',
        yaxis_title='Power Output (W)',
        width=1000,
        height=500,
        legend=dict(x=0.02, y=0.98)
    )
    
    return fig

# Example usage (uncomment when data is available):
# fig = plot_comparison_timeseries(df_initial, df_final)
# fig.show()

In [ ]:
# === TEMPLATE: Monthly Improvement Bar Chart ===

def plot_monthly_improvement(df_before, df_after, datetime_col='Datetime', power_col='average_dc_power'):
    """
    Show improvement by month.
    """
    df_before['Month'] = pd.to_datetime(df_before[datetime_col]).dt.month_name()
    df_after['Month'] = pd.to_datetime(df_after[datetime_col]).dt.month_name()
    
    monthly_before = df_before.groupby('Month')[power_col].sum()
    monthly_after = df_after.groupby('Month')[power_col].sum()
    
    comparison = pd.DataFrame({
        'Before': monthly_before,
        'After': monthly_after
    }).reset_index()
    
    comparison['Improvement_%'] = ((comparison['After'] - comparison['Before']) / comparison['Before'] * 100).round(2)
    
    fig = px.bar(
        comparison,
        x='Month',
        y=['Before', 'After'],
        barmode='group',
        title='Monthly Energy Production: Before vs After Optimization',
        labels={'value': 'Total Energy (Wh)', 'Month': 'Month'}
    )
    
    fig.update_layout(width=1000, height=500)
    
    return fig, comparison

# Example usage (uncomment when data is available):
# fig, monthly_comparison = plot_monthly_improvement(df_initial, df_final)
# fig.show()
# monthly_comparison

---

## 📋 Summary of Exported Files

| File | Description | Use Case |
|------|-------------|----------|
| `optimal_angles_monthly.csv` | Monthly tilt recommendations | Simple fixed-angle implementation |
| `optimal_angles_hourly.csv` | Hourly recommendations by month | Manual adjustment systems |
| `optimal_angles_daily_noon.csv` | Daily noon solar positions | Tracking system reference |

### Key Recommendations for Goa (15.49°N):

| Season | Months | Recommended Tilt | Notes |
|--------|--------|------------------|-------|
| Winter | Nov-Feb | **30°** | Sun is lower, steeper angle |
| Summer | May-Aug | **0-5°** | Sun nearly overhead |
| Equinox | Mar-Apr, Sep-Oct | **15°** | Equal to latitude |

**Panel Azimuth**: Always face **South (180°)** in Northern Hemisphere